# GeneTropica Phase 14 — Molecular Dynamics Simulation

**Three-drug mechanism comparison on DENV NS5 RdRp (PDB 5CCV)**

| Drug | Consensus Rank | Mechanism | Purpose |
|------|---------------|-----------|----------|
| Celecoxib | #1 (0.6846) | COX-2 inhibitor | Pipeline's top prediction |
| Methotrexate | #3 (0.4930) | DHFR / host-directed | Tests indirect mechanism |
| Dasabuvir | #17 (0.3758) | Non-nucleoside RdRp | Direct polymerase binder |

**Protocol:** 50 ns all-atom MD per drug using GROMACS + ACPYPE (GAFF)

**Runtime estimate:** 3–9 days total (T4 GPU)

---
Russell Young — British School Jakarta

In [ ]:
# ============================================================
# Cell 1: Install GROMACS and dependencies (~3 min)
# ============================================================
import subprocess, sys, os

print('Installing GROMACS...')
!apt-get update -qq
!apt-get install -y -qq gromacs > /dev/null 2>&1

print('Installing ACPYPE for ligand parametrisation...')
!pip install -q acpype

print('Installing analysis tools...')
!pip install -q MDAnalysis matplotlib numpy

# Verify installation
!gmx --version | head -5
!acpype --help 2>&1 | head -3
print('\n=== Installation complete ===')

In [ ]:
# ============================================================
# Cell 2: Upload input files (select ALL 8 files)
# ============================================================
from google.colab import files
import os

WORKDIR = '/content/md_simulation'
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)

print('Please upload ALL 8 files:')
print('  1. protein_5CCV.pdb')
print('  2. celecoxib_docked.mol2')
print('  3. methotrexate_docked.mol2')
print('  4. dasabuvir_docked.mol2')
print('  5. em.mdp')
print('  6. nvt.mdp')
print('  7. npt.mdp')
print('  8. md.mdp')
print()

uploaded = files.upload()

# Verify all files
REQUIRED = [
    'protein_5CCV.pdb',
    'celecoxib_docked.mol2', 'methotrexate_docked.mol2', 'dasabuvir_docked.mol2',
    'em.mdp', 'nvt.mdp', 'npt.mdp', 'md.mdp',
]
missing = [f for f in REQUIRED if not os.path.exists(f)]
if missing:
    print(f'\n*** MISSING FILES: {missing} ***')
    print('Please re-upload the missing files.')
else:
    print(f'\n=== All {len(REQUIRED)} files uploaded successfully ===')
    for f in sorted(os.listdir('.')):
        print(f'  {f} ({os.path.getsize(f):,} bytes)')

In [ ]:
# ============================================================
# Cell 3: Verify GPU is available
# ============================================================
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    for line in result.stdout.split('\n')[:10]:
        print(line)
    print('\n=== GPU detected — MD will use GPU acceleration ===')
else:
    print('*** WARNING: No GPU detected! ***')
    print('Go to Runtime > Change runtime type > T4 GPU > Save')
    print('Then re-run from Cell 1.')

In [ ]:
# ============================================================
# Cell 4: Helper functions (shared by all 3 drugs)
# ============================================================
import matplotlib.pyplot as plt
import numpy as np
import os, subprocess, shutil, glob

WORKDIR = '/content/md_simulation'

def run_cmd(cmd, label='', check=True):
    """Run a shell command, print output on failure."""
    if label:
        print(f'  [{label}]')
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and result.returncode != 0:
        print(f'  STDERR: {result.stderr[:2000]}')
        raise RuntimeError(f'Command failed: {cmd}')
    return result


def plot_xvg(xvg_path, title, xlabel, ylabel, save_path):
    """Parse GROMACS .xvg file and plot."""
    x, y = [], []
    with open(xvg_path) as f:
        for line in f:
            if line.startswith(('#', '@')):
                continue
            parts = line.split()
            if len(parts) >= 2:
                x.append(float(parts[0]))
                y.append(float(parts[1]))
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(x, y, linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.show()
    print(f'  Saved: {save_path}')


def prepare_system(drug_name):
    """Prepare GROMACS system for one drug.
    
    Steps: clean protein -> pdb2gmx -> ACPYPE ligand -> combine -> solvate -> ions
    """
    drug_dir = os.path.join(WORKDIR, drug_name)
    os.makedirs(drug_dir, exist_ok=True)
    os.chdir(drug_dir)
    
    mol2_file = os.path.join(WORKDIR, f'{drug_name}_docked.mol2')
    protein_pdb = os.path.join(WORKDIR, 'protein_5CCV.pdb')
    
    print(f'\n{"="*60}')
    print(f'  PREPARING: {drug_name.upper()}')
    print(f'{"="*60}')
    
    # --- Step 1: Process protein with pdb2gmx (AMBER99SB-ILDN) ---
    print('\n[1/7] Processing protein with pdb2gmx...')
    run_cmd(
        f'echo 1 | gmx pdb2gmx -f {protein_pdb} -o protein.gro '
        f'-p topol.top -ignh -ff amber99sb-ildn -water tip3p',
        'pdb2gmx'
    )
    
    # --- Step 2: Parametrise ligand with ACPYPE ---
    print('\n[2/7] Parametrising ligand with ACPYPE (GAFF)...')
    run_cmd(
        f'acpype -i {mol2_file} -c bcc -n 0 -a gaff2',
        'acpype'
    )
    
    # Find the ACPYPE output directory
    acpype_dirs = glob.glob(f'{drug_name}*.acpype') + glob.glob('*.acpype')
    if not acpype_dirs:
        raise FileNotFoundError('ACPYPE output directory not found')
    acpype_dir = acpype_dirs[0]
    print(f'  ACPYPE output: {acpype_dir}')
    
    # Copy ACPYPE GROMACS files
    lig_itp = glob.glob(os.path.join(acpype_dir, '*_GMX.itp'))
    lig_gro = glob.glob(os.path.join(acpype_dir, '*_GMX.gro'))
    
    if not lig_itp or not lig_gro:
        # Try alternative naming
        lig_itp = glob.glob(os.path.join(acpype_dir, '*.itp'))
        lig_gro = glob.glob(os.path.join(acpype_dir, '*.gro'))
        # Filter out posre files
        lig_itp = [f for f in lig_itp if 'posre' not in f.lower()]
    
    shutil.copy(lig_itp[0], 'ligand.itp')
    shutil.copy(lig_gro[0], 'ligand.gro')
    
    # Also copy atomtypes if present
    at_files = glob.glob(os.path.join(acpype_dir, '*atomtypes*itp'))
    has_atomtypes = False
    if at_files:
        shutil.copy(at_files[0], 'ligand_atomtypes.itp')
        has_atomtypes = True
    
    # --- Step 3: Combine protein + ligand ---
    print('\n[3/7] Combining protein + ligand...')
    
    # Read protein .gro
    with open('protein.gro') as f:
        prot_lines = f.readlines()
    
    # Read ligand .gro
    with open('ligand.gro') as f:
        lig_lines = f.readlines()
    
    # Combine: header + (protein atoms + ligand atoms) + box
    prot_natoms = int(prot_lines[1].strip())
    lig_natoms = int(lig_lines[1].strip())
    total_atoms = prot_natoms + lig_natoms
    
    with open('complex.gro', 'w') as f:
        f.write(f'Protein-ligand complex: {drug_name}\n')
        f.write(f'{total_atoms}\n')
        # Protein atoms (skip header 2 lines, skip box last line)
        for line in prot_lines[2:2+prot_natoms]:
            f.write(line)
        # Ligand atoms
        for line in lig_lines[2:2+lig_natoms]:
            f.write(line)
        # Box from protein
        f.write(prot_lines[-1])
    
    print(f'  Combined: {prot_natoms} protein + {lig_natoms} ligand = {total_atoms} atoms')
    
    # --- Step 4: Update topology ---
    print('\n[4/7] Updating topology...')
    
    with open('topol.top') as f:
        top_content = f.read()
    
    # Determine ligand residue name from the .itp file
    lig_resname = 'LIG'
    with open('ligand.itp') as f:
        for line in f:
            if line.strip() and not line.startswith(';') and not line.startswith('['):
                parts = line.split()
                if len(parts) >= 5:
                    lig_resname = parts[3] if len(parts) > 3 else 'LIG'
                    break
    
    # Insert ligand atomtypes and itp include after forcefield include
    lines = top_content.split('\n')
    new_lines = []
    ff_included = False
    for line in lines:
        new_lines.append(line)
        if 'forcefield.itp' in line and not ff_included:
            if has_atomtypes:
                new_lines.append('#include "ligand_atomtypes.itp"')
            new_lines.append('#include "ligand.itp"')
            ff_included = True
    
    # Add ligand molecule at end of [ molecules ] section
    new_lines.append(f'{lig_resname}     1')
    
    with open('topol.top', 'w') as f:
        f.write('\n'.join(new_lines))
    
    print(f'  Ligand residue name: {lig_resname}')
    print(f'  Topology updated with ligand includes')
    
    # --- Step 5: Solvate ---
    print('\n[5/7] Defining box and solvating...')
    run_cmd(
        'gmx editconf -f complex.gro -o box.gro '
        '-c -d 1.2 -bt dodecahedron',
        'editconf'
    )
    run_cmd(
        'gmx solvate -cp box.gro -cs spc216.gro '
        '-o solvated.gro -p topol.top',
        'solvate'
    )
    
    # --- Step 6: Add ions ---
    print('\n[6/7] Adding ions to neutralise system...')
    run_cmd(
        f'gmx grompp -f {os.path.join(WORKDIR, "em.mdp")} '
        f'-c solvated.gro -p topol.top -o ions.tpr -maxwarn 5',
        'grompp-ions'
    )
    run_cmd(
        'echo SOL | gmx genion -s ions.tpr -o system.gro '
        '-p topol.top -pname NA -nname CL -neutral',
        'genion'
    )
    
    # --- Step 7: Create index groups ---
    print('\n[7/7] Creating index groups...')
    # Create custom index with Protein_LIG group
    run_cmd(
        'echo -e "1 | 13\nq" | gmx make_ndx -f system.gro -o index.ndx',
        'make_ndx',
        check=False  # may have different group numbers
    )
    
    # Alternative: create index via script
    # Try to identify the right groups
    result = run_cmd('echo q | gmx make_ndx -f system.gro', check=False)
    ndx_output = result.stdout + result.stderr
    print(f'  Available index groups:')
    for line in ndx_output.split('\n'):
        if line.strip().startswith(('0 ', '1 ', '2 ', '3 ', '4 ', '5 ',
                                    '6 ', '7 ', '8 ', '9 ', '10', '11',
                                    '12', '13', '14', '15', '16', '17',
                                    '18', '19', '20')):
            print(f'    {line.strip()}')
    
    # Create a proper Protein_LIG group
    # Group 1 = Protein, find the ligand group number
    # Write a selection command
    with open('make_ndx_input.txt', 'w') as f:
        f.write('1 | 13\nname 21 Protein_LIG\nq\n')
    run_cmd(
        'gmx make_ndx -f system.gro -o index.ndx < make_ndx_input.txt',
        'make_ndx-final',
        check=False
    )
    
    print(f'\n=== {drug_name.upper()} system prepared ===')
    print(f'  Working directory: {drug_dir}')
    return drug_dir


def run_equilibration(drug_name):
    """Run energy minimisation + NVT + NPT equilibration."""
    drug_dir = os.path.join(WORKDIR, drug_name)
    os.chdir(drug_dir)
    
    print(f'\n{"="*60}')
    print(f'  EQUILIBRATING: {drug_name.upper()}')
    print(f'{"="*60}')
    
    # --- Energy Minimisation ---
    print('\n[EM] Energy minimisation...')
    run_cmd(
        f'gmx grompp -f {os.path.join(WORKDIR, "em.mdp")} '
        f'-c system.gro -p topol.top -o em.tpr -maxwarn 5',
        'grompp-em'
    )
    run_cmd('gmx mdrun -v -deffnm em -nb gpu', 'mdrun-em')
    
    # Plot EM energy
    run_cmd('echo "10\n0" | gmx energy -f em.edr -o em_potential.xvg', 'energy-em', check=False)
    if os.path.exists('em_potential.xvg'):
        plot_xvg('em_potential.xvg',
                 f'{drug_name} — Energy Minimisation',
                 'Step', 'Potential Energy (kJ/mol)',
                 f'em_energy_{drug_name}.png')
    
    # --- NVT Equilibration ---
    print('\n[NVT] NVT equilibration (100 ps, 300K)...')
    run_cmd(
        f'gmx grompp -f {os.path.join(WORKDIR, "nvt.mdp")} '
        f'-c em.gro -r em.gro -p topol.top -o nvt.tpr -maxwarn 5',
        'grompp-nvt'
    )
    run_cmd('gmx mdrun -deffnm nvt -nb gpu', 'mdrun-nvt')
    
    # Plot NVT temperature
    run_cmd('echo "16\n0" | gmx energy -f nvt.edr -o nvt_temp.xvg', 'energy-nvt', check=False)
    if os.path.exists('nvt_temp.xvg'):
        plot_xvg('nvt_temp.xvg',
                 f'{drug_name} — NVT Temperature',
                 'Time (ps)', 'Temperature (K)',
                 f'nvt_temp_{drug_name}.png')
    
    # --- NPT Equilibration ---
    print('\n[NPT] NPT equilibration (100 ps, 300K, 1 bar)...')
    run_cmd(
        f'gmx grompp -f {os.path.join(WORKDIR, "npt.mdp")} '
        f'-c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -o npt.tpr -maxwarn 5',
        'grompp-npt'
    )
    run_cmd('gmx mdrun -deffnm npt -nb gpu', 'mdrun-npt')
    
    # Plot NPT density
    run_cmd('echo "24\n0" | gmx energy -f npt.edr -o npt_density.xvg', 'energy-npt', check=False)
    if os.path.exists('npt_density.xvg'):
        plot_xvg('npt_density.xvg',
                 f'{drug_name} — NPT Density',
                 'Time (ps)', 'Density (kg/m³)',
                 f'npt_density_{drug_name}.png')
    
    # Quick RMSD sanity check
    run_cmd(
        'echo "4\n4" | gmx rms -s em.tpr -f npt.xtc -o rmsd_equil.xvg -tu ps',
        'rmsd-equil',
        check=False
    )
    if os.path.exists('rmsd_equil.xvg'):
        plot_xvg('rmsd_equil.xvg',
                 f'{drug_name} — Equilibration RMSD (backbone)',
                 'Time (ps)', 'RMSD (nm)',
                 f'rmsd_equil_{drug_name}.png')
    
    print(f'\n=== {drug_name.upper()} equilibration complete ===')


def run_production_full(drug_name):
    """Run full 50 ns production MD in one shot."""
    drug_dir = os.path.join(WORKDIR, drug_name)
    os.chdir(drug_dir)
    
    print(f'\n{"="*60}')
    print(f'  PRODUCTION (50 ns): {drug_name.upper()}')
    print(f'{"="*60}')
    
    run_cmd(
        f'gmx grompp -f {os.path.join(WORKDIR, "md.mdp")} '
        f'-c npt.gro -t npt.cpt -p topol.top -o md.tpr -maxwarn 5',
        'grompp-md'
    )
    print('  Starting 50 ns production run...')
    print('  This will take 24-72 hours on T4 GPU.')
    run_cmd('gmx mdrun -deffnm md -nb gpu -v', 'mdrun-production')
    
    print(f'\n=== {drug_name.upper()} production complete ===')


def run_production_chunked(drug_name, chunk_ns=10, total_ns=50):
    """Run production in chunks for Colab Free tier."""
    drug_dir = os.path.join(WORKDIR, drug_name)
    os.chdir(drug_dir)
    
    n_chunks = total_ns // chunk_ns
    steps_per_chunk = int(chunk_ns * 1e6 / 2)  # dt=0.002 ps
    
    print(f'\n{"="*60}')
    print(f'  PRODUCTION (CHUNKED {n_chunks}x{chunk_ns}ns): {drug_name.upper()}')
    print(f'{"="*60}')
    
    for chunk in range(n_chunks):
        print(f'\n--- Chunk {chunk+1}/{n_chunks} ({chunk*chunk_ns}-{(chunk+1)*chunk_ns} ns) ---')
        
        if chunk == 0:
            # First chunk: start from NPT
            # Write a modified md.mdp for chunk length
            with open(os.path.join(WORKDIR, 'md.mdp')) as f:
                mdp_content = f.read()
            mdp_content = mdp_content.replace(
                'nsteps              = 25000000',
                f'nsteps              = {steps_per_chunk}'
            )
            with open('md_chunk.mdp', 'w') as f:
                f.write(mdp_content)
            
            run_cmd(
                f'gmx grompp -f md_chunk.mdp '
                f'-c npt.gro -t npt.cpt -p topol.top -o md.tpr -maxwarn 5',
                f'grompp-chunk{chunk+1}'
            )
        else:
            # Extend from checkpoint
            run_cmd(
                f'gmx convert-tpr -s md.tpr -extend {chunk_ns * 1000} -o md.tpr',
                f'extend-chunk{chunk+1}'
            )
        
        print(f'  Running chunk {chunk+1}... (~5-14 hours on T4)')
        run_cmd(
            'gmx mdrun -deffnm md -nb gpu -v -cpi md.cpt',
            f'mdrun-chunk{chunk+1}'
        )
        
        print(f'  Chunk {chunk+1} complete!')
        
        # Save checkpoint to Drive after each chunk
        try:
            drive_dir = f'/content/drive/MyDrive/GeneTropica_MD/{drug_name}'
            os.makedirs(drive_dir, exist_ok=True)
            for f in ['md.cpt', 'md.xtc', 'md.edr', 'md.log']:
                if os.path.exists(f):
                    shutil.copy(f, drive_dir)
            print(f'  Saved checkpoint to Google Drive: {drive_dir}')
        except Exception as e:
            print(f'  (Drive save skipped: {e})')
    
    print(f'\n=== {drug_name.upper()} chunked production complete ===')


def package_results(drug_name):
    """Package key result files into a tar.gz for download."""
    drug_dir = os.path.join(WORKDIR, drug_name)
    os.chdir(drug_dir)
    
    result_files = ['md.xtc', 'md.tpr', 'md.gro', 'md.edr', 'topol.top',
                    'md.cpt', 'md.log', 'index.ndx',
                    'em.gro', 'npt.gro', 'system.gro']
    # Also include equilibration plots
    result_files += glob.glob('*.png')
    
    existing = [f for f in result_files if os.path.exists(f)]
    
    tar_name = f'md_results_{drug_name}.tar.gz'
    tar_path = os.path.join(WORKDIR, tar_name)
    
    file_list = ' '.join(existing)
    run_cmd(f'tar -czf {tar_path} {file_list}', f'package-{drug_name}')
    
    size_mb = os.path.getsize(tar_path) / 1e6
    print(f'\n  Packaged: {tar_name} ({size_mb:.1f} MB)')
    print(f'  Contains: {len(existing)} files')
    return tar_path

print('=== Helper functions loaded ===')

In [ ]:
# ============================================================
# Cell 5: (Optional) Mount Google Drive for backup
# ============================================================
# Uncomment the lines below to enable Google Drive backup.
# Each chunk checkpoint will be saved automatically.

# from google.colab import drive
# drive.mount('/content/drive')
# print('Google Drive mounted. Checkpoints will be saved to:')
# print('  /content/drive/MyDrive/GeneTropica_MD/')

In [ ]:
# ============================================================
# Cell 6: Prepare all 3 drug systems (~15 min total)
# ============================================================
os.chdir(WORKDIR)

DRUGS = ['celecoxib', 'methotrexate', 'dasabuvir']

for drug in DRUGS:
    prepare_system(drug)

print('\n' + '='*60)
print('  ALL 3 SYSTEMS PREPARED SUCCESSFULLY')
print('='*60)

In [ ]:
# ============================================================
# Cell 7: Equilibrate all 3 systems (EM + NVT + NPT) (~90 min)
# ============================================================
# Verify plots after each drug:
# - EM energy: should drop steeply then flatten
# - NVT temp: should oscillate around 300 K
# - NPT density: should stabilise near 1000 kg/m³
# - Equil RMSD: should rise then level off < 5 Å

for drug in DRUGS:
    run_equilibration(drug)

print('\n' + '='*60)
print('  ALL 3 SYSTEMS EQUILIBRATED')
print('  Check the plots above before proceeding!')
print('='*60)

In [ ]:
# ============================================================
# Cell 8: CHOOSE YOUR PRODUCTION MODE
# ============================================================
#
# >>> OPTION A: Full 50 ns (Colab Pro / long session)
# Runs all 3 drugs back-to-back. Total ~3-9 days.
# Use this if you have Colab Pro or a stable connection.
#
# >>> OPTION B: Chunked 5x10 ns (Colab Free — RECOMMENDED)
# Runs each drug in 5 chunks of 10 ns.
# Each chunk ~5-14 hours. Download results between drugs.
#
# Set your choice below:

PRODUCTION_MODE = 'chunked'  # 'full' or 'chunked'

print(f'Production mode: {PRODUCTION_MODE}')
if PRODUCTION_MODE == 'chunked':
    print('Each drug: 5 chunks of 10 ns each.')
    print('Recommended: download each drug\'s results before starting the next.')
else:
    print('Full 50 ns runs. Ensure stable connection.')

In [ ]:
# ============================================================
# Cell 9: CELECOXIB — Production MD
# ============================================================
# Consensus #1 (score 0.6846, ML 0.7026)
# COX-2 selective inhibitor — pipeline's top recommendation
# Question: Does the pipeline's best prediction actually bind NS5 RdRp?

if PRODUCTION_MODE == 'full':
    run_production_full('celecoxib')
else:
    run_production_chunked('celecoxib', chunk_ns=10, total_ns=50)

In [ ]:
# ============================================================
# Cell 10: Package and download CELECOXIB results
# ============================================================
cel_tar = package_results('celecoxib')

# Download
try:
    files.download(cel_tar)
    print('Download started for md_results_celecoxib.tar.gz')
except Exception as e:
    print(f'Auto-download failed: {e}')
    print(f'Manual download: use Files sidebar > {cel_tar}')

# Also save to Drive if mounted
try:
    drive_dest = '/content/drive/MyDrive/GeneTropica_MD/'
    os.makedirs(drive_dest, exist_ok=True)
    shutil.copy(cel_tar, drive_dest)
    print(f'Saved to Google Drive: {drive_dest}')
except:
    pass

print('\n>>> Download celecoxib results before running methotrexate! <<<')

In [ ]:
# ============================================================
# Cell 11: METHOTREXATE — Production MD
# ============================================================
# Consensus #3 (score 0.4930, ML 0.4834)
# DHFR inhibitor — host-directed mechanism (nucleotide pool depletion)
# Question: Can a drug with an indirect mechanism also bind the polymerase?

if PRODUCTION_MODE == 'full':
    run_production_full('methotrexate')
else:
    run_production_chunked('methotrexate', chunk_ns=10, total_ns=50)

In [ ]:
# ============================================================
# Cell 12: Package and download METHOTREXATE results
# ============================================================
mtx_tar = package_results('methotrexate')

try:
    files.download(mtx_tar)
    print('Download started for md_results_methotrexate.tar.gz')
except Exception as e:
    print(f'Auto-download failed: {e}')
    print(f'Manual download: use Files sidebar > {mtx_tar}')

try:
    drive_dest = '/content/drive/MyDrive/GeneTropica_MD/'
    shutil.copy(mtx_tar, drive_dest)
    print(f'Saved to Google Drive: {drive_dest}')
except:
    pass

print('\n>>> Download methotrexate results before running dasabuvir! <<<')

In [ ]:
# ============================================================
# Cell 13: DASABUVIR — Production MD
# ============================================================
# Consensus #17 (score 0.3758)
# Non-nucleoside NS5B RdRp palm-site inhibitor
# Published dengue in vitro data (PMID 37632595)
# Question: Does cross-family RdRp conservation enable HCV→dengue binding?

if PRODUCTION_MODE == 'full':
    run_production_full('dasabuvir')
else:
    run_production_chunked('dasabuvir', chunk_ns=10, total_ns=50)

In [ ]:
# ============================================================
# Cell 14: Package and download DASABUVIR results
# ============================================================
das_tar = package_results('dasabuvir')

try:
    files.download(das_tar)
    print('Download started for md_results_dasabuvir.tar.gz')
except Exception as e:
    print(f'Auto-download failed: {e}')
    print(f'Manual download: use Files sidebar > {das_tar}')

try:
    drive_dest = '/content/drive/MyDrive/GeneTropica_MD/'
    shutil.copy(das_tar, drive_dest)
    print(f'Saved to Google Drive: {drive_dest}')
except:
    pass

print('\n=== ALL THREE SIMULATIONS COMPLETE ===')
print('\nYou should now have 3 files:')
print('  1. md_results_celecoxib.tar.gz')
print('  2. md_results_methotrexate.tar.gz')
print('  3. md_results_dasabuvir.tar.gz')
print('\nExtract these to your local genetropica folder and run Part 2.')

## Disconnection Recovery

If Colab disconnects during a production run, follow these steps:

1. Re-run **Cell 1** (reinstall GROMACS)
2. Re-run **Cell 2** (re-upload 8 files)
3. Re-run **Cell 3** (GPU check)
4. Re-run **Cell 4** (helper functions)
5. **Skip** Cell 6 and 7 (preparation and equilibration) if checkpoint exists
6. Jump to the **drug cell** where disconnection occurred
7. The chunked mode will resume from the last checkpoint automatically

If using full mode and disconnected, switch to chunked mode and restart the current drug.

In [ ]:
# ============================================================
# Cell 16: Recovery — Resume from checkpoint after disconnection
# ============================================================
# 1. After re-running Cells 1-4, set the drug name below
# 2. Run this cell to re-prepare the system files
# 3. Then run the drug's production cell (9, 11, or 13)

RESUME_DRUG = 'celecoxib'  # Change to the drug that was interrupted

drug_dir = os.path.join(WORKDIR, RESUME_DRUG)

if os.path.exists(os.path.join(drug_dir, 'md.cpt')):
    print(f'Checkpoint found for {RESUME_DRUG}!')
    print(f'  md.cpt exists — production can resume.')
    print(f'  Run the production cell for {RESUME_DRUG} to continue.')
elif os.path.exists(os.path.join(drug_dir, 'npt.gro')):
    print(f'Equilibration complete for {RESUME_DRUG}.')
    print(f'  Run the production cell to start from scratch.')
else:
    print(f'No checkpoint found for {RESUME_DRUG}.')
    print(f'  Re-run Cell 6 (prepare) and Cell 7 (equilibrate) first.')
    print(f'  Then run the production cell.')